# WSMTE — 02_feature_engineering.ipynb
**LOCAL PC** | Steps 9–17 from DATA_PIPELINE.md  
Wavelet denoising → technical indicators → split → scale → sliding windows → targets.

**Prerequisites**: `data/processed/merged_data.csv` must exist (run `01_data_prep.ipynb` first).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import pywt
import joblib
import json
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from config.config import CONFIG

## Load merged_data.csv

In [ ]:
df = pd.read_csv(CONFIG['processed_data_dir'] + 'merged_data.csv')
df['date'] = pd.to_datetime(df['date']).dt.date
df = df.sort_values('date').reset_index(drop=True)
print(f'Loaded merged_data.csv: {df.shape}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Columns: {list(df.columns)}')

## Step 9 — Coif3 Wavelet Denoising
> Applied to Open, High, Low, Close, Volume independently BEFORE any indicator computation.

In [ ]:
def coif3_denoise(series):
    """
    Coif3, level=1, soft thresholding.
    Universal threshold via median absolute deviation.
    Returns denoised array of same length as input.
    """
    coeffs = pywt.wavedec(series, CONFIG['wavelet'], level=CONFIG['wavelet_level'])
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    threshold = sigma * np.sqrt(2 * np.log(len(series)))
    coeffs[1:] = [pywt.threshold(c, threshold, mode=CONFIG['wavelet_mode'])
                  for c in coeffs[1:]]
    return pywt.waverec(coeffs, CONFIG['wavelet'])[:len(series)]

for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
    df[f'{col}_d'] = coif3_denoise(df[col].values)
    print(f'Denoised {col}: std_orig={df[col].std():.4f}  std_denoised={df[f"{col}_d"].std():.4f}')

## Step 10 — Technical Indicators on DENOISED Prices
> All indicators use `_d` columns, NOT raw prices.

In [ ]:
# ── RSI(14) ──────────────────────────────────────────────────────
def compute_rsi(series, period=None):
    if period is None:
        period = CONFIG['rsi_period']
    delta = series.diff()
    gain  = delta.where(delta > 0, 0.0).rolling(period).mean()
    loss  = (-delta.where(delta < 0, 0.0)).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df['RSI_d'] = compute_rsi(df['Close_d'])
print(f'RSI_d range: {df["RSI_d"].min():.1f} to {df["RSI_d"].max():.1f}')

In [ ]:
# ── MACD histogram: EMA(12) - EMA(26) ──────────────────────────────────
ema_fast = df['Close_d'].ewm(span=CONFIG['ema_fast'], adjust=False).mean()
ema_slow = df['Close_d'].ewm(span=CONFIG['ema_slow'], adjust=False).mean()
df['MACD_d'] = ema_fast - ema_slow
print(f'MACD_d range: {df["MACD_d"].min():.4f} to {df["MACD_d"].max():.4f}')

In [ ]:
# ── Bollinger Band Width: (Upper - Lower) / Middle ───────────────────────
sma20 = df['Close_d'].rolling(CONFIG['bb_period']).mean()
std20 = df['Close_d'].rolling(CONFIG['bb_period']).std()
df['BB_width_d'] = (CONFIG['bb_std'] * 2 * std20) / sma20
print(f'BB_width_d range: {df["BB_width_d"].min():.4f} to {df["BB_width_d"].max():.4f}')

In [ ]:
# ── ROC(5): 5-day rate of change ───────────────────────────────────────
df['ROC_d'] = df['Close_d'].pct_change(periods=CONFIG['roc_period']) * 100
print(f'ROC_d range: {df["ROC_d"].min():.4f} to {df["ROC_d"].max():.4f}')

## Step 11 — Build Final Feature Dataframe
> Drop warmup rows (26 days for MACD EMA(26) convergence).

In [ ]:
FEATURE_COLUMNS = CONFIG['feature_columns']
# Expected: ['Close_d','Volume_d','RSI_d','MACD_d','BB_width_d','ROC_d',
#            'polarity_company','polarity_market','subjectivity']

df = df.dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)
print(f'After warmup drop ({CONFIG["warmup_days"]} days): {df.shape}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')

df[['date'] + FEATURE_COLUMNS + ['Close']].to_csv(
    CONFIG['processed_data_dir'] + 'final_dataset.csv', index=False)
print('Saved → data/processed/final_dataset.csv')

## Step 12 — Verify No Missing Values

In [ ]:
assert df[FEATURE_COLUMNS].isnull().sum().sum() == 0, \
    'Missing values found — fix pipeline before continuing'
print(f'No missing values ✓  |  Final shape: {df.shape}')
print(df[FEATURE_COLUMNS].describe().round(4))

## Step 13 — Temporal Split (70 / 15 / 15, NO shuffling)

In [ ]:
n = len(df)
train_end = int(n * CONFIG['train_ratio'])
val_end   = int(n * (CONFIG['train_ratio'] + CONFIG['val_ratio']))

train_df = df.iloc[:train_end].reset_index(drop=True)
val_df   = df.iloc[train_end:val_end].reset_index(drop=True)
test_df  = df.iloc[val_end:].reset_index(drop=True)

print(f'Total: {n}  |  Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')
assert train_df['date'].max() < val_df['date'].min(), 'Temporal leakage: train/val overlap!'
assert val_df['date'].max() < test_df['date'].min(),  'Temporal leakage: val/test overlap!'
print(f'Train: {train_df["date"].min()} → {train_df["date"].max()}')
print(f'Val:   {val_df["date"].min()} → {val_df["date"].max()}')
print(f'Test:  {test_df["date"].min()} → {test_df["date"].max()}')
print('No temporal leakage ✓')

## Step 14 — MinMaxScaler (fit on train ONLY)

In [ ]:
scaler = MinMaxScaler()  # scales each feature independently

train_scaled = scaler.fit_transform(train_df[FEATURE_COLUMNS])
val_scaled   = scaler.transform(val_df[FEATURE_COLUMNS])
test_scaled  = scaler.transform(test_df[FEATURE_COLUMNS])

# Sanity check: train must be [0,1] since scaler was fit on it
assert train_scaled.max() <= 1.0 and train_scaled.min() >= 0.0, 'Scaler sanity check failed!'
print(f'Train scaled: min={train_scaled.min():.4f}  max={train_scaled.max():.4f}  ✓')
print(f'Val   scaled: min={val_scaled.min():.4f}  max={val_scaled.max():.4f}')
print(f'Test  scaled: min={test_scaled.min():.4f}  max={test_scaled.max():.4f}')

joblib.dump(scaler, CONFIG['processed_data_dir'] + 'scaler.pkl')
print('Saved → data/processed/scaler.pkl')

## Step 15 — Sliding Windows (window_size=5)

In [ ]:
WINDOW_SIZE = CONFIG['window_size']

def create_windows(scaled_data, raw_close, window_size=WINDOW_SIZE):
    """
    For each position i in [window_size, len(scaled_data)):
      X[i]     = scaled_data[i-window_size : i]        shape [5, 9]
      y_clf[i] = 1 if raw_close[i] > raw_close[i-1] else 0
      y_reg[i] = scaled_data[i][0]                     (scaled Close_d)
    """
    X, y_clf, y_reg = [], [], []
    for i in range(window_size, len(scaled_data)):
        X.append(scaled_data[i - window_size : i])
        y_clf.append(1 if raw_close[i] > raw_close[i - 1] else 0)
        y_reg.append(scaled_data[i][0])  # index 0 = Close_d
    return (np.array(X, dtype=np.float32),
            np.array(y_clf, dtype=np.int32),
            np.array(y_reg, dtype=np.float32))

X_train, y_clf_train, y_reg_train = create_windows(train_scaled, train_df['Close'].values)
X_val,   y_clf_val,   y_reg_val   = create_windows(val_scaled,   val_df['Close'].values)
X_test,  y_clf_test,  y_reg_test  = create_windows(test_scaled,  test_df['Close'].values)

print(f'X_train: {X_train.shape}  |  X_val: {X_val.shape}  |  X_test: {X_test.shape}')
assert X_train.shape[1] == WINDOW_SIZE and X_train.shape[2] == CONFIG['n_features'], \
    f'Expected window shape [{WINDOW_SIZE}, {CONFIG["n_features"]}], got {X_train.shape[1:]}'
print(f'Window shape [{WINDOW_SIZE}, {CONFIG["n_features"]}] ✓')

In [ ]:
# Save all arrays
arrays = {
    'X_train': X_train,     'X_val': X_val,     'X_test': X_test,
    'y_clf_train': y_clf_train, 'y_clf_val': y_clf_val, 'y_clf_test': y_clf_test,
    'y_reg_train': y_reg_train, 'y_reg_val': y_reg_val, 'y_reg_test': y_reg_test,
}
for name, arr in arrays.items():
    np.save(CONFIG['processed_data_dir'] + f'{name}.npy', arr)
    print(f'Saved {name}.npy  shape={arr.shape}  dtype={arr.dtype}')

## Step 16 — Class Imbalance Check

In [ ]:
counts = Counter(y_clf_train)
ratio    = max(counts.values()) / min(counts.values())
up_pct   = counts[1] / len(y_clf_train) * 100
down_pct = counts[0] / len(y_clf_train) * 100
print(f'Up: {up_pct:.1f}%  |  Down: {down_pct:.1f}%  |  Ratio: {ratio:.2f}')

if ratio > CONFIG['imbalance_threshold']:  # > 1.5 (60:40)
    weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_clf_train)
    class_weight_dict = {0: float(weights[0]), 1: float(weights[1])}
    print(f'Imbalanced — applying class weights: {class_weight_dict}')
else:
    class_weight_dict = None
    print('Balanced — no class weighting needed')

with open(CONFIG['processed_data_dir'] + 'class_weights.json', 'w') as f:
    json.dump(class_weight_dict, f)
print('Saved → data/processed/class_weights.json')

## Summary

In [ ]:
print('=' * 55)
print('Feature engineering complete.')
print(f'  final_dataset.csv : {df.shape}')
print(f'  X_train           : {X_train.shape}')
print(f'  X_val             : {X_val.shape}')
print(f'  X_test            : {X_test.shape}')
print(f'  y_clf_train       : {y_clf_train.shape}  labels={set(y_clf_train)}')
print(f'  class_weights     : {class_weight_dict}')
print('=' * 55)
print('Ready for Day 3 — model implementation.')